# Lab 19c: Tools with Built-In AI Gateway

> **Disclaimer:** The API interactions in this lab are for educational purposes only, intended to explain how these components work behind the scenes. They are **not** meant as templates for automation. Some features demonstrated here may currently only be available through the Azure portal and do not yet have automation templates or official SDK support.

In **Lab 7B**, you learned how to add MCP tools to the Foundry Tool Catalog and use them with agents. Those tools connected **directly** to MCP servers.

This lab builds on that foundation by showing how to route tool traffic through the **Built-In AI Gateway** (APIM). This gives you centralized monitoring, rate limiting, and governance for all tool calls - the same benefits you get for model inference.

## What You'll Build

| Step | What | Why |
|------|------|-----|
| 1 | Create a **Direct Tool** | Same as Lab 7B - tool calls go straight to MCP server |
| 2 | Create an **AI Gateway Tool** | Route tool calls through APIM for governance |
| 3 | Use both with an **Agent** | See both approaches work identically |
| 4 | Apply **APIM Policies** | Demonstrate governance benefits |

## Direct vs AI Gateway Tools

| Aspect | Direct Tool (Lab 7B) | AI Gateway Tool (This Lab) |
|--------|----------------------|----------------------------|
| **Target URL** | `https://mcp-server.example.com` | `https://{apim}.azure-api.net/tool-...` |
| **Traffic Flow** | Agent → MCP Server | Agent → APIM → MCP Server |
| **Monitoring** | None (or per-tool logging) | Centralized APIM Analytics |
| **Governance** | Manual per-tool | Unified APIM policies |

## Prerequisites

- Lab 1A completed (Landing Zone with APIM)
- Lab 1B completed (Foundry project spoke)
- Lab 7B recommended (Foundry Tool Catalog basics)

---

## Step 1: Setup and Configuration

In [12]:
import os
import json
from IPython.display import display, Markdown

from tools_helpers import (
    load_env, get_subscription_id,
    create_tool_connection, delete_tool_connection,
    register_tool_in_apim, list_tool_connections,
    compare_tool_routing,
    mask_resource_name, mask_subscription, mask_url
)

# Load environment
load_env('/workspaces/getting-started-with-foundry/.env')

# Configuration
SUBSCRIPTION = get_subscription_id()
SPOKE_ACCOUNT = os.environ.get('SPOKE_ACCOUNT', '')
SPOKE_PROJECT = os.environ.get('SPOKE_PROJECT', '')
SPOKE_RG = "foundry-child-1"

# APIM configuration (from landing zone)
APIM_URL = os.environ.get('APIM_URL', '')
APIM_NAME = APIM_URL.split('//')[1].split('.')[0] if APIM_URL and '//' in APIM_URL else ''
APIM_RG = "foundry-lz-parent"

# Model configuration
APIM_CONNECTION = os.environ.get('APIM_CONNECTION', '')
MODEL_NAME = os.environ.get('MODEL_NAME', 'gpt-4.1-mini')
GATEWAY_MODEL = f"{APIM_CONNECTION}/{MODEL_NAME}"

PROJECT_ENDPOINT = f"https://{SPOKE_ACCOUNT}.services.ai.azure.com/api/projects/{SPOKE_PROJECT}"

# MCP Server URL for demonstration
MCP_SERVER_URL = "https://learn.microsoft.com/api/mcp"

print(f"Subscription: {mask_subscription(SUBSCRIPTION)}")
print(f"Project: {mask_resource_name(SPOKE_PROJECT)}")
print(f"Account: {mask_resource_name(SPOKE_ACCOUNT)}")
print(f"APIM: {mask_resource_name(APIM_NAME)}")
print(f"MCP Server: {MCP_SERVER_URL}")

Subscription: da22829d...
Project: project-******
Account: foundry-spoke-******
APIM: foundry-lz-apim-******
MCP Server: https://learn.microsoft.com/api/mcp


---

## Step 2: Create a Direct Tool Connection

First, let's create a **direct tool connection**. This is the simplest approach - the tool connection points directly to the MCP server URL. When an agent uses this tool, requests go straight to the MCP server.

```
Agent → MCP Server (direct)
```

In [13]:
DIRECT_TOOL_NAME = "mslearn-direct"

print("Creating Direct Tool Connection")
print("=" * 50)

direct_result = create_tool_connection(
    subscription=SUBSCRIPTION,
    rg=SPOKE_RG,
    account_name=SPOKE_ACCOUNT,
    project_name=SPOKE_PROJECT,
    connection_name=DIRECT_TOOL_NAME,
    target_url=MCP_SERVER_URL  # Points directly to MCP server
)

if "error" in direct_result:
    print(f"Error: {direct_result['error']}")
else:
    print(f"Connection: {direct_result['connection_name']}")
    print(f"Target URL: {direct_result['target_url']}")
    print(f"AI Gateway: {direct_result['is_gateway_routed']}")
    
    display(Markdown(f"""
### Direct Tool Created

The tool `{DIRECT_TOOL_NAME}` now exists in the Foundry Tool Catalog.

- **Target**: `{MCP_SERVER_URL}`
- **Routing**: Direct (no APIM)
- **Monitoring**: None centralized

Tool calls will go directly from the agent to the MCP server.
"""))

Creating Direct Tool Connection
Connection: mslearn-direct
Target URL: https://learn.microsoft.com/api/mcp
AI Gateway: False



### Direct Tool Created

The tool `mslearn-direct` now exists in the Foundry Tool Catalog.

- **Target**: `https://learn.microsoft.com/api/mcp`
- **Routing**: Direct (no APIM)
- **Monitoring**: None centralized

Tool calls will go directly from the agent to the MCP server.


---

## Step 3: Create an AI Gateway Tool

Now let's create the **same tool but routed through the AI Gateway** (APIM). This is a two-step process:

1. **Register in APIM** - Create an API with `type: mcp` that proxies to the MCP server
2. **Create Connection** - Point the Foundry tool connection to the APIM endpoint

```
Agent → APIM (AI Gateway) → MCP Server
```

In [14]:
GATEWAY_TOOL_NAME = "mslearn-gateway"

print("Step 3a: Register MCP server in APIM")
print("=" * 50)

# Register the tool in APIM (creates backend + API)
apim_result = register_tool_in_apim(
    subscription=SUBSCRIPTION,
    apim_rg=APIM_RG,
    apim_name=APIM_NAME,
    project_name=SPOKE_PROJECT,
    tool_url=MCP_SERVER_URL,
    display_name="MS Learn MCP (Gateway)"
)

if "error" in apim_result:
    print(f"Error: {apim_result['error']}")
    GATEWAY_ENDPOINT = None
else:
    GATEWAY_ENDPOINT = apim_result['gateway_endpoint']
    print(f"APIM API: {mask_resource_name(apim_result['api_name'])}")
    print(f"Gateway Endpoint: {mask_url(GATEWAY_ENDPOINT)}")
    print(f"Backend URL: {apim_result['backend_url']}")

Step 3a: Register MCP server in APIM
APIM API: tool-project-h6cmx3-learn-microsoft-com-api-mcp
Gateway Endpoint: https://foundry-lz-apim-lagivk.azure-api.net/tool-project-h6cmx3-learn-microsoft-com-api-mcp
Backend URL: https://learn.microsoft.com/api/mcp


In [15]:
print("\nStep 3b: Create Foundry tool connection")
print("=" * 50)

if GATEWAY_ENDPOINT:
    gateway_result = create_tool_connection(
        subscription=SUBSCRIPTION,
        rg=SPOKE_RG,
        account_name=SPOKE_ACCOUNT,
        project_name=SPOKE_PROJECT,
        connection_name=GATEWAY_TOOL_NAME,
        target_url=GATEWAY_ENDPOINT  # Points to APIM, not direct!
    )
    
    if "error" in gateway_result:
        print(f"Error: {gateway_result['error']}")
    else:
        print(f"Connection: {gateway_result['connection_name']}")
        print(f"Target URL: {gateway_result['target_url']}")
        print(f"AI Gateway: {gateway_result['is_gateway_routed']}")
        
        display(Markdown(f"""
### AI Gateway Tool Created

The tool `{GATEWAY_TOOL_NAME}` now exists in the Foundry Tool Catalog.

- **Target**: `{GATEWAY_ENDPOINT}`
- **Backend**: `{MCP_SERVER_URL}`
- **Routing**: Via APIM (AI Gateway enabled)
- **Monitoring**: Centralized in APIM Analytics

Tool calls will flow through APIM before reaching the MCP server.
"""))
else:
    print("Skipped - APIM registration failed")


Step 3b: Create Foundry tool connection
Connection: mslearn-gateway
Target URL: https://foundry-lz-apim-lagivk.azure-api.net/tool-project-h6cmx3-learn-microsoft-com-api-mcp
AI Gateway: True



### AI Gateway Tool Created

The tool `mslearn-gateway` now exists in the Foundry Tool Catalog.

- **Target**: `https://foundry-lz-apim-lagivk.azure-api.net/tool-project-h6cmx3-learn-microsoft-com-api-mcp`
- **Backend**: `https://learn.microsoft.com/api/mcp`
- **Routing**: Via APIM (AI Gateway enabled)
- **Monitoring**: Centralized in APIM Analytics

Tool calls will flow through APIM before reaching the MCP server.


---

## Step 4: Compare Both Tools

Now we have two tools pointing to the **same MCP server** but with different routing:

In [16]:
# List and compare all tool connections
project_tools = list_tool_connections(SUBSCRIPTION, SPOKE_RG, SPOKE_ACCOUNT, SPOKE_PROJECT)
comparison = compare_tool_routing(project_tools)

display(Markdown("## Tool Routing Comparison"))

print(f"Total tools: {comparison['total']}")
print(f"  - Direct routing: {comparison['direct_count']}")
print(f"  - AI Gateway routing: {comparison['gateway_count']}")

if project_tools:
    table_md = """
| Tool Name | Routing | Target Endpoint |
|-----------|---------|------------------|
"""
    for tool in project_tools:
        routing = "AI Gateway" if tool.is_gateway_routed else "Direct"
        masked_endpoint = mask_url(tool.endpoint)
        endpoint_display = masked_endpoint[:55] + "..." if len(masked_endpoint) > 55 else masked_endpoint
        table_md += f"| {tool.name} | {routing} | `{endpoint_display}` |\n"
    
    display(Markdown(table_md))

## Tool Routing Comparison

Total tools: 2
  - Direct routing: 1
  - AI Gateway routing: 1



| Tool Name | Routing | Target Endpoint |
|-----------|---------|------------------|
| mslearn-direct | Direct | `https://learn.microsoft.com/api/mcp` |
| mslearn-gateway | AI Gateway | `https://foundry-lz-apim-lagivk.azure-api.net/tool-proje...` |


---

## Step 5: Use Both Tools with an Agent

Let's create an agent that can use **both tools**. From the agent's perspective, they work identically - it doesn't know (or care) that one routes through APIM.

In [17]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# Retrieve both tool connections
print("Retrieving tool connections from catalog...")

try:
    direct_tool = project_client.connections.get(DIRECT_TOOL_NAME)
    print(f"  Direct tool: {direct_tool.name} → {direct_tool.target[:20]}...")
    DIRECT_AVAILABLE = True
except Exception as e:
    print(f"  Direct tool not found: {e}")
    DIRECT_AVAILABLE = False

try:
    gateway_tool = project_client.connections.get(GATEWAY_TOOL_NAME)
    print(f"  Gateway tool: {gateway_tool.name} → {gateway_tool.target[:20]}...")
    GATEWAY_AVAILABLE = True
except Exception as e:
    print(f"  Gateway tool not found: {e}")
    GATEWAY_AVAILABLE = False

Retrieving tool connections from catalog...
  Direct tool: mslearn-direct → https://learn.micros...
  Gateway tool: mslearn-gateway → https://foundry-lz-a...


In [18]:
AGENT_NAME = "docs-assistant-dual-tools"

# Build tools list based on what's available
agent_tools = []

if DIRECT_AVAILABLE:
    agent_tools.append({
        "type": "mcp",
        "server_label": "mslearn-direct",
        "server_url": direct_tool.target,
        "project_connection_id": direct_tool.id,
        "require_approval": "never"
    })

if GATEWAY_AVAILABLE:
    agent_tools.append({
        "type": "mcp",
        "server_label": "mslearn-gateway",
        "server_url": gateway_tool.target,
        "project_connection_id": gateway_tool.id,
        "require_approval": "never"
    })

if agent_tools:
    agent = project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=GATEWAY_MODEL,
            instructions="""You help users find information from Microsoft Learn documentation.
            
Use the mslearn-gateway tool (AI Gateway routed) to search documentation.""",
            tools=agent_tools
        ),
    )
    
    print(f"Agent created: {agent.name} v{agent.version}")
    print(f"\nConfigured tools: {len(agent_tools)}")
    for t in agent_tools:
        print(f"  - {t['server_label']}")
else:
    print("No tools available - run Steps 2 and 3 first")

Agent created: docs-assistant-dual-tools v1

Configured tools: 2
  - mslearn-direct
  - mslearn-gateway


In [19]:
# Test the agent
if agent_tools:
    openai_client = project_client.get_openai_client()
    
    test_query = "What is Azure Functions?"
    
    print(f"User: {test_query}")
    print("\n" + "=" * 50)
    print("Processing (gateway tool calls route through APIM)...")
    print("=" * 50 + "\n")
    
    response = openai_client.responses.create(
        input=test_query,
        extra_body={
            "agent": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference"
            }
        }
    )
    
    print(f"Agent: {response.output_text}")
else:
    print("Skipped - No agent created")

User: What is Azure Functions?

Processing (gateway tool calls route through APIM)...

Agent: Azure Functions is a serverless computing solution that enables you to build robust applications with less code, less infrastructure, and at lower costs. It allows you to run your code without worrying about deploying or maintaining servers, as the cloud infrastructure automatically provides and manages the necessary resources. You focus on writing the code that matters most in your preferred programming language, and Azure Functions handles the rest.

Key aspects of Azure Functions include:

- **Event-driven**: It supports a comprehensive set of triggers and bindings that connect your function to other services without extra code.
- **Common scenarios**: Processing file uploads, real-time stream and event processing, running AI inference, scheduled tasks, scalable web APIs, serverless workflows, responding to database changes, and creating reliable messaging systems.
- **Supported languages**

---

## Step 6: AI Gateway Governance Benefits

Now that the gateway tool is registered in APIM, you gain powerful governance capabilities **without changing the agent configuration**.

### Available APIM Policies for MCP Tools

| Policy | Use Case |
|--------|----------|
| **Rate Limiting** | Prevent tool abuse, control costs |
| **IP Filtering** | Restrict tool access by network |
| **Request Validation** | Validate MCP request payloads |
| **Response Caching** | Cache repeated tool queries |
| **Logging** | Detailed request/response logging |

In [20]:
# Generate portal URLs for managing policies
if APIM_NAME and 'apim_result' in dir() and apim_result and 'api_name' in apim_result:
    api_name = apim_result['api_name']
    
    apim_policy_url = (
        f"https://portal.azure.com/#@/resource/subscriptions/{SUBSCRIPTION}"
        f"/resourceGroups/{APIM_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}"
        f"/apis/{api_name}/policies"
    )
    
    apim_analytics_url = (
        f"https://portal.azure.com/#@/resource/subscriptions/{SUBSCRIPTION}"
        f"/resourceGroups/{APIM_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}"
        f"/analytics"
    )
    
    display(Markdown(f"""
## APIM Management URLs

### Edit Tool Policies
Add rate limiting, IP filtering, caching, etc.:

[Open Tool Policies in APIM]({apim_policy_url})

### View Analytics
Monitor all tool calls (requests, latency, errors):

[Open APIM Analytics]({apim_analytics_url})

### Example Policy: Rate Limiting

Add this to the `<inbound>` section to limit to 10 calls per minute:

```xml
<rate-limit calls="10" renewal-period="60" />
```
"""))


## APIM Management URLs

### Edit Tool Policies
Add rate limiting, IP filtering, caching, etc.:

[Open Tool Policies in APIM](https://portal.azure.com/#@/resource/subscriptions/da22829d-006e-4d97-9fd4-9678afa702d9/resourceGroups/foundry-lz-parent/providers/Microsoft.ApiManagement/service/foundry-lz-apim-lagivk/apis/tool-project-h6cmx3-learn-microsoft-com-api-mcp/policies)

### View Analytics
Monitor all tool calls (requests, latency, errors):

[Open APIM Analytics](https://portal.azure.com/#@/resource/subscriptions/da22829d-006e-4d97-9fd4-9678afa702d9/resourceGroups/foundry-lz-parent/providers/Microsoft.ApiManagement/service/foundry-lz-apim-lagivk/analytics)

### Example Policy: Rate Limiting

Add this to the `<inbound>` section to limit to 10 calls per minute:

```xml
<rate-limit calls="10" renewal-period="60" />
```


---

## Summary

### What We Built

| Tool | Target | Governance |
|------|--------|------------|
| `mslearn-direct` | MCP Server (direct) | None |
| `mslearn-gateway` | APIM → MCP Server | Full APIM policies |

### Key Takeaways

1. **Same MCP Server, Different Routes** - Both tools connect to the same backend, but one routes through APIM

2. **Agent Doesn't Know** - The agent uses both tools identically; routing is transparent

3. **APIM Governance Benefits**:
   - Centralized monitoring and analytics
   - Rate limiting and quota management
   - Request/response logging
   - Unified policies for models AND tools

### When to Use AI Gateway for Tools

| Use Direct Tools | Use AI Gateway Tools |
|------------------|----------------------|
| Development/testing | Production workloads |
| Simple single-agent apps | Enterprise multi-agent systems |
| No governance requirements | Need monitoring, rate limiting |
| Minimal latency critical | Governance > latency tradeoff acceptable |

---

## Cleanup

In [21]:
print("Cleanup")
print("=" * 50)

# Delete agent
if 'agent' in dir() and agent_tools:
    try:
        project_client.agents.delete_version(agent_name=AGENT_NAME, agent_version=agent.version)
        print(f"Agent deleted: {AGENT_NAME}")
    except Exception as e:
        print(f"Agent cleanup: {e}")

# Delete direct tool connection
direct_delete = delete_tool_connection(
    subscription=SUBSCRIPTION,
    rg=SPOKE_RG,
    account_name=SPOKE_ACCOUNT,
    project_name=SPOKE_PROJECT,
    connection_name=DIRECT_TOOL_NAME
)
print(f"Direct tool deleted: {direct_delete.get('success', direct_delete.get('error'))}")

# Delete gateway tool connection
gateway_delete = delete_tool_connection(
    subscription=SUBSCRIPTION,
    rg=SPOKE_RG,
    account_name=SPOKE_ACCOUNT,
    project_name=SPOKE_PROJECT,
    connection_name=GATEWAY_TOOL_NAME
)
print(f"Gateway tool deleted: {gateway_delete.get('success', gateway_delete.get('error'))}")

print("\nNote: APIM API not deleted (can be reused)")
if 'apim_result' in dir() and apim_result and 'api_name' in apim_result:
    print(f"To remove: delete API '{mask_resource_name(apim_result['api_name'])}' from APIM portal")

print("\nCleanup complete")

Cleanup


Agent deleted: docs-assistant-dual-tools
Direct tool deleted: True
Gateway tool deleted: True

Note: APIM API not deleted (can be reused)
To remove: delete API 'tool-project-h6cmx3-learn-microsoft-com-api-mcp' from APIM portal

Cleanup complete
